In [1]:
!pip install pymupdf4llm

In [2]:
import pymupdf4llm

In [3]:
PDF_PATH = "dummy_book.pdf"

In [4]:
def parse_pdf_to_markdown(path: str) -> str:
    return pymupdf4llm.to_markdown(path)

In [5]:
parsed_document =  parse_pdf_to_markdown(PDF_PATH)

=== Document parser messages ===
Using RapidOCR for OCR processing.



In [6]:
parsed_document

'# **A Practitioner\'s Guide to Language Models** \n\nAn end-to-end tour from tokens to production serving. \n\n# **Chapter 1: Tokenization** \n\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled. \n\nEarly systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare words into reusable fragments.

# CHUNKS

### RECURSIVE TEXT SPLITTER

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # Maximum characters per chunk
    chunk_overlap=150  # Overlap between chunks
)

In [8]:
chunks = text_splitter.split_text(parsed_document)

In [9]:
chunks

["# **A Practitioner's Guide to Language Models** \n\nAn end-to-end tour from tokens to production serving. \n\n# **Chapter 1: Tokenization** \n\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.",
 'Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare words into reusable fragments.

In [10]:
for i in chunks:
    print(i)
    print("*****"*40)

# **A Practitioner's Guide to Language Models** 

An end-to-end tour from tokens to production serving. 

# **Chapter 1: Tokenization** 

Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.
********************************************************************************************************************************************************************************************************
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the 

### MARKDOWN TEXT SPLITTERS

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Define headers to split on
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]



markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)


docs = markdown_splitter.split_text(parsed_document)

In [12]:
from langchain_text_splitters import MarkdownTextSplitter

# Create the splitter
splitter = MarkdownTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [13]:
chunks = splitter.split_text(parsed_document)

In [14]:
chunks[2]

'```\ndef bpe_merges(words, num_merges):\n    """Learn BPE merges from a list of space-separated symbol strings."""\n    from collections import Counter\n    merges = []\n    for _ in range(num_merges):\n        pairs = Counter()\n        for w in words:\n            symbols = w.split()\n            for a, b in zip(symbols, symbols[1:]):\n                pairs[(a, b)] += 1          # count adjacent pairs\n        if not pairs:\n            break\n```\n\n```\n        best = max(pairs, key=pairs.get)    # most frequent pair\n        merges.append(best)\n        bigram = " ".join(best)\n        words = [w.replace(bigram, "".join(best)) for w in words]\n    return merges'

In [15]:
for i in chunks:
    print(chunks)
    print("****"*40)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [16]:
for i, doc in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print("Content:")
    print(doc.page_content)


--- Chunk 1 ---
Metadata: {'Header 1': "**A Practitioner's Guide to Language Models**"}
Content:
An end-to-end tour from tokens to production serving.

--- Chunk 2 ---
Metadata: {'Header 1': '**Chapter 1: Tokenization**'}
Content:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.  
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenizatio

## PARTITION OF DATA AS PER THEIR RESPECTIVE DATA-TYPE

In [17]:
# CREATING AN SIMPLE DATA CLASS NAMED ELEMENT , THIS WILL BE USED AS A CONTAINER TO STORE PARTITION INFORMATION LIKE WHAT KIND IT IS 
# ACTUAL CONTENT ETC.


from dataclasses import dataclass, field, asdict 

@dataclass
class Element:
    kind: str       # "prose" | "code" | "table"
    text: str
    chapter: str = ""
    section: str = ""

In [19]:
#FUNCTION TO READ DOCUMENT AND PATITION IT ON BASIS OF ITS DATA TYPE.

def partition_markdown(md: str): #FILE LOADED AS MARKDOWN FILE WITH THE HELP OF LOADER
    lines = md.split("\n")  #SPLITTING THE MARKDOWN FILE LINE BY LINE.
    elements: List[Element] = [] #creates an empty list that will store the extracted document parts (e.g., text, code blocks, tables) as Element objects.
    # initialize variables to keep track of the current chapter and section headings while parsing the document.
    chapter = ""
    section = ""

    #STARTING LOOP
    i, n = 0, len(lines)  # initiated two variables i=0 and n= number of lines.
    while i < n:
        stripped = lines[i].strip() # stripping first line

    #CAPTURING HIERARCHY
    #Lets first detect  Markdown headings and updating the current chapter/section context.
        if stripped.startswith("#"): #Checks whether the current line is a Markdown heading (#, ##, ###, etc.).
            level = len(stripped) - len(stripped.lstrip("#")) #Counts how many # symbols are at the beginning to determine the heading level.
            title = stripped.lstrip("#").strip().strip("*").strip()#extracting title.
            if level == 1 or title.lower().startswith(("chapter", "appendix")):
                chapter = title
                section = ""
            else:
                section = title
            i += 1
            continue


        #SEPERATING CODE PART
        if stripped.startswith("```"):
            code_lines = [] #EMPTY LIST TO STORE CODE.
            i += 1 # MOVING TO NEXT LINE AS CODE WILL START FROM NEXT LINE.
            while i < n and not lines[i].strip().startswith("```"): #KEEP READING LINE UNTILL YOU FIND "```" AND STORE THEM IN CODE_LINES
                code_lines.append(lines[i])
                i += 1
            i += 1
            code_text = "\n".join(code_lines)   #COMBINING ALL THE CODE LINES INTO ONE STRING.
            elements.append(Element("code", code_text, chapter, section)) # add new element of type code to the list we created above with type code.
            continue



       #SEPARATING TABLE PART.
        if stripped.startswith("|"):
            table_lines = [] # CREATES AN EMPTY LIST TO STORE ALL THE TABLE ROWS.
            while i < n and lines[i].strip().startswith("|"): #KEEP READING LINES AS LONG AS THEY START WITH '|'
                table_lines.append(lines[i].strip()) #Adds each table row to the list and moves to the next line.
                i += 1
            elements.append(Element("table", "\n".join(table_lines), chapter, section)) # ADD THE NEW ELEMENT OF TYPE TABLE TO THE LIST WE JUST CREATED.
            continue

        #to handle edge cases
        if stripped.startswith(">") and elements and elements[-1].kind == "code":
            cont = stripped.lstrip(">").strip().strip("`")
            elements[-1].text += "\n" + cont
            i += 1
            continue



        #SEPARATING PROSE [TEXTUAL] PART.
        if stripped:
            prose_lines = [] #CREATE AN LIST TO STORE PROSE
            while i < n and lines[i].strip() and not lines[i].strip().startswith(("#", "```", "|",">")): #CONTINUE COLLECTING LINES TILL YOU REACHED THE
                #END OF THE DOCUMENT, LINE IS NOT EMPTY, LINE IS NOT A HEADING , CODE BLOCK ETC.
                prose_lines.append(lines[i].strip()) #APPEND TO PROSE_LINE
                i += 1
            elements.append(Element("prose", " ".join(prose_lines), chapter, section)) #ADD NEW ELEMENT OF TYPE PROSE.
            continue
 
        i += 1
 
    return elements

In [20]:
heading = "#### ***Background"

level = len(heading) - len(heading.lstrip("#"))
title = heading.lstrip("#").strip().strip("*").strip()

print(level)
print(title)

4
Background


In [21]:
markdown_doc = """
# Chapter 1: Introduction
This is just a normal paragraph of text.
## Section 1.1: Math
Here is a simple table:

| A | B |
|---|---|
| 1 | 2 |

## Section 1.2: Code
Here is a Python function:
```python
def hello():
    print("world")
```"""

results = partition_markdown(markdown_doc)

In [22]:
results

[Element(kind='prose', text='This is just a normal paragraph of text.', chapter='Chapter 1: Introduction', section=''),
 Element(kind='prose', text='Here is a simple table:', chapter='Chapter 1: Introduction', section='Section 1.1: Math'),
 Element(kind='table', text='| A | B |\n|---|---|\n| 1 | 2 |', chapter='Chapter 1: Introduction', section='Section 1.1: Math'),
 Element(kind='prose', text='Here is a Python function:', chapter='Chapter 1: Introduction', section='Section 1.2: Code'),
 Element(kind='code', text='def hello():\n    print("world")', chapter='Chapter 1: Introduction', section='Section 1.2: Code')]

In [23]:
#WRITTING IN PROPER WAY
for chunk in results:
    print(f"TYPE: {chunk.kind.upper()}")
    print(f"LOCATION: Chapter '{chunk.chapter}' > Section '{chunk.section}'")
    print(f"CONTENT:\n{chunk.text}")
    print("-" * 40)

TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Introduction' > Section ''
CONTENT:
This is just a normal paragraph of text.
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Introduction' > Section 'Section 1.1: Math'
CONTENT:
Here is a simple table:
----------------------------------------
TYPE: TABLE
LOCATION: Chapter 'Chapter 1: Introduction' > Section 'Section 1.1: Math'
CONTENT:
| A | B |
|---|---|
| 1 | 2 |
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Introduction' > Section 'Section 1.2: Code'
CONTENT:
Here is a Python function:
----------------------------------------
TYPE: CODE
LOCATION: Chapter 'Chapter 1: Introduction' > Section 'Section 1.2: Code'
CONTENT:
def hello():
    print("world")
----------------------------------------


In [24]:
partitioned_document=partition_markdown(parsed_document)

In [25]:
for chunk in partitioned_document:
    print(f"TYPE: {chunk.kind.upper()}")
    print(f"LOCATION: Chapter '{chunk.chapter}' > Section '{chunk.section}'")
    print(f"CONTENT:\n{chunk.text}")
    print("-" * 40)

TYPE: PROSE
LOCATION: Chapter 'A Practitioner's Guide to Language Models' > Section ''
CONTENT:
An end-to-end tour from tokens to production serving.
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Tokenization' > Section ''
CONTENT:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Tokenization' > Section ''
CONTENT:
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input

In [26]:
# ABOVE CODE ONE  PROBLEM , AS I SPLITTED IT ON BASIS OF LINE , PARTITION THAT SHOULD BE COMBINED ARE BECOMING NEW NEW PARTITIONS.

#CHECKING IF THE PREVIOUS PART IS ALSO OF SAME TYPE AS NEXT PART AND SAME CHAPTER , SECTION , MERGE THE PARTS.

In [27]:
merged_chunks = []

for chunk in partitioned_document:
    if (
        merged_chunks
        and chunk.kind in ("prose", "code")
        and merged_chunks[-1].kind == chunk.kind
        and merged_chunks[-1].chapter == chunk.chapter
        and merged_chunks[-1].section == chunk.section
    ):
        # Extend the previous chunk
        merged_chunks[-1].text += "\n" + chunk.text
    else:
        merged_chunks.append(chunk)

# Print merged output
for chunk in merged_chunks:
    print(f"TYPE: {chunk.kind.upper()}")
    print(f"LOCATION: Chapter '{chunk.chapter}' > Section '{chunk.section}'")
    print(f"CONTENT:\n{chunk.text}")
    print("-" * 40)

TYPE: PROSE
LOCATION: Chapter 'A Practitioner's Guide to Language Models' > Section ''
CONTENT:
An end-to-end tour from tokens to production serving.
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Tokenization' > Section ''
CONTENT:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these

In [28]:
merged_chunks

[Element(kind='prose', text='An end-to-end tour from tokens to production serving.', chapter="A Practitioner's Guide to Language Models", section=''),
 Element(kind='prose', text='Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.\nEarly systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare

## ROUTING AND SPLITTING.

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [30]:
#CREATING SPLITTERS FOR RESPECTIVE TYPES OF DATA
prose_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Set small for testing; increase to ~800 for production
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""]
)


# B. Code Splitter: Uses LangChain's Python-aware rules (looks for def, class, etc.)
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size= 1500,
    chunk_overlap=0,          
)


def split_code(code_text: str):
    return code_splitter.split_text(code_text)


#KEEPING TABLES AS THEY ARE AS I HAVE SMALL TABLES

In [31]:
# Create a Python-aware code splitter
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=150,
    chunk_overlap=0,
)

def split_code(code_text: str):
    return code_splitter.split_text(code_text)



sample_code = """
class Calculator:
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b


def greet(name):
    print(f"Hello, {name}!")


def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)


def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
"""

In [32]:
chunks = split_code(sample_code)

In [33]:
for chunk in (chunks):
    print(chunk)
    print("*******"*10)

class Calculator:
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b
**********************************************************************
def greet(name):
    print(f"Hello, {name}!")


def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)
**********************************************************************
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
**********************************************************************


In [34]:
# ROUTING AND SPLITTING
def route_and_split(element: Element):
    """Route each element to the appropriate splitter."""

    if element.kind == "prose":
        return prose_splitter.split_text(element.text)

    elif element.kind == "code":
        return split_code(element.text)

    elif element.kind == "table":
        # Keep table as a single chunk
        return [element.text]

    return [element.text]

In [35]:
#CREATING CHUNKS
chunked_documents = []

for element in merged_chunks:
    chunks = route_and_split(element) #creating chunk

    for chunk_text in chunks: # using the chunk to find its meta data from the element and appending it to chunk.
        chunked_documents.append(
            {
                "kind": element.kind,
                "chapter": element.chapter,
                "section": element.section,
                "text": chunk_text,
            }
        )

In [36]:
chunked_documents

[{'kind': 'prose',
  'chapter': "A Practitioner's Guide to Language Models",
  'section': '',
  'text': 'An end-to-end tour from tokens to production serving.'},
 {'kind': 'prose',
  'chapter': 'Chapter 1: Tokenization',
  'section': '',
  'text': 'Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.'},
 {'kind': 'prose',
  'chapter': 'Chapter 1: Tokenization',
  'section': '',
  'text': 'Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the 

In [37]:
# looking at chunked documents.
len(chunked_documents)

201

In [38]:
for chunk in chunked_documents:  # or partitioned_document

    print("\n" + "=" * 100)
    print(f"TYPE    : {chunk['kind'].upper()}")
    print(f"CHAPTER : {chunk['chapter']}")
    print(f"SECTION : {chunk['section']}")

    if chunk["kind"] == "code":
        print("\nCODE:")
        print("-" * 100)
        print(chunk["text"])  # preserves indentation
        print("-" * 100)

    elif chunk["kind"] == "table":
        print("\nTABLE:")
        print("-" * 100)
        print(chunk["text"])
        print("-" * 100)

    else:
        print("\nTEXT:")
        print(chunk["text"])

    print("=" * 100)


TYPE    : PROSE
CHAPTER : A Practitioner's Guide to Language Models
SECTION : 

TEXT:
An end-to-end tour from tokens to production serving.

TYPE    : PROSE
CHAPTER : Chapter 1: Tokenization
SECTION : 

TEXT:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.

TYPE    : PROSE
CHAPTER : Chapter 1: Tokenization
SECTION : 

TEXT:
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit bet

### CONTEXTUAL RETRIEVAL

In [39]:
import ollama 

CONTEXT_PROMPT = """<document>
{document}
</document>
Here is the chunk we want to situate within the whole document
<chunk>
{chunk}
</chunk>
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk. 
Answer only with the succinct context and nothing else."""

MODEL= "gemma4:31b-cloud"

#GENERATING SUMMARY.
def generate_context(document: str, chunk: str, model: str = MODEL) -> str:
    prompt = CONTEXT_PROMPT.format(document=document, chunk=chunk)
    resp = ollama.generate(
        model=model,
        prompt=prompt,
        options={"temperature": 0.0},   
    )

    return resp.response.strip()



In [40]:
enriched_documents = []

print(f"Starting enrichment for {len(chunked_documents)} chunks...")

for i, item in enumerate(chunked_documents):
    original_chunk_text = item['text']
    
    print(f"Processing chunk {i + 1}/{len(chunked_documents)}...")
    generated_context = generate_context(document=parsed_document, chunk=original_chunk_text)
    
    # 2. Create a copy of the dictionary so we don't mutate the original data
    enriched_item = item.copy()
    
    # 3. Save the raw context just in case you need it for debugging later
    enriched_item['context'] = generated_context
    
    # 4. Glue the context to the front of the original chunk text
    enriched_item['enriched_text'] = f"{generated_context}\n{original_chunk_text}"
    
    # 5. Append it to our new list
    enriched_documents.append(enriched_item)

print("Enrichment complete!")

# You can now take 'enriched_documents' and embed/store the 'enriched_text' field into your vector database.

Starting enrichment for 201 chunks...
Processing chunk 1/201...
Processing chunk 2/201...
Processing chunk 3/201...
Processing chunk 4/201...
Processing chunk 5/201...
Processing chunk 6/201...
Processing chunk 7/201...
Processing chunk 8/201...
Processing chunk 9/201...
Processing chunk 10/201...
Processing chunk 11/201...
Processing chunk 12/201...
Processing chunk 13/201...
Processing chunk 14/201...
Processing chunk 15/201...
Processing chunk 16/201...
Processing chunk 17/201...
Processing chunk 18/201...
Processing chunk 19/201...
Processing chunk 20/201...
Processing chunk 21/201...
Processing chunk 22/201...
Processing chunk 23/201...
Processing chunk 24/201...
Processing chunk 25/201...
Processing chunk 26/201...
Processing chunk 27/201...
Processing chunk 28/201...
Processing chunk 29/201...
Processing chunk 30/201...
Processing chunk 31/201...
Processing chunk 32/201...
Processing chunk 33/201...
Processing chunk 34/201...
Processing chunk 35/201...
Processing chunk 36/201...

In [43]:
enriched_documents[3]

{'kind': 'prose',
 'chapter': 'Chapter 1: Tokenization',
 'section': '1.1 Byte-Pair Encoding',
 'text': "Byte-Pair Encoding, or BPE, starts from individual characters and repeatedly merges the most frequent adjacent pair into a new symbol. After thousands of merges, common sequences like 'ing' or 'tion' become single tokens, while rare words remain splittable into known pieces. This gives a fixed vocabulary that can still represent any string.\nThe algorithm is simple enough to implement in a few lines. The core loop counts adjacent pairs, finds the most frequent, and records the merge.",
 'context': 'This chunk is from Chapter 1 (Tokenization) of "A Practitioner\'s Guide to Language Models," specifically Section 1.1, explaining the mechanics and purpose of the Byte-Pair Encoding (BPE) algorithm for subword tokenization.',
 'enriched_text': 'This chunk is from Chapter 1 (Tokenization) of "A Practitioner\'s Guide to Language Models," specifically Section 1.1, explaining the mechanics an

In [44]:
enriched_documents

[{'kind': 'prose',
  'chapter': "A Practitioner's Guide to Language Models",
  'section': '',
  'text': 'An end-to-end tour from tokens to production serving.',
  'context': 'Subtitle of the book "A Practitioner\'s Guide to Language Models," introducing the scope of the text from tokenization to deployment.',
  'enriched_text': 'Subtitle of the book "A Practitioner\'s Guide to Language Models," introducing the scope of the text from tokenization to deployment.\nAn end-to-end tour from tokens to production serving.'},
 {'kind': 'prose',
  'chapter': 'Chapter 1: Tokenization',
  'section': '',
  'text': 'Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affect

## EMBEDDINGS

In [162]:
texts = [r["enriched_text"] for r in enriched_documents]

In [163]:
texts

['Subtitle of the book "A Practitioner\'s Guide to Language Models," introducing the scope of the text from tokenization to deployment.\nAn end-to-end tour from tokens to production serving.',
 'This chunk is the introductory paragraph of Chapter 1: Tokenization in "A Practitioner\'s Guide to Language Models," establishing the fundamental role and downstream impact of converting text into tokens.\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.',
 'This chunk is from Chapter 1 (Tokenization) of "A Practitioner\'s Guide to Language Models," specifically discussing the evolution of tokenization methods from whitespace

In [164]:
texts[1]

'This chunk is the introductory paragraph of Chapter 1: Tokenization in "A Practitioner\'s Guide to Language Models," establishing the fundamental role and downstream impact of converting text into tokens.\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.'

In [165]:
embeddings = ollama.embed(model="qwen3-embedding:8b", input=texts).embeddings 

In [166]:
embeddings

[[0.022380073,
  0.008870771,
  0.00045484124,
  -0.03831518,
  0.029102068,
  -0.015978223,
  -0.007171008,
  -0.012853572,
  0.006265102,
  5.5928656e-05,
  -0.05667398,
  0.000991815,
  0.05347854,
  -0.0088357935,
  0.042000692,
  -0.0044553336,
  0.022435477,
  0.0050108735,
  -0.027001267,
  0.0055391644,
  -0.003426053,
  0.06810386,
  0.000601341,
  0.0036604644,
  0.0014519388,
  0.021702085,
  0.010921793,
  -0.004923523,
  0.015870111,
  0.0044911285,
  -0.02451192,
  0.017330103,
  0.00029914427,
  0.036988616,
  -0.037679914,
  0.026670886,
  -0.022146443,
  0.045603976,
  0.024780255,
  -0.004897253,
  -0.006641747,
  -0.009855803,
  0.018133279,
  -0.019518588,
  -0.017769925,
  0.002094885,
  -0.022370674,
  -0.006955989,
  -0.0071105678,
  -0.010376085,
  0.00048404513,
  -0.022836706,
  0.025677463,
  0.007159489,
  0.01054258,
  -0.022783877,
  -0.018927548,
  -0.0027541188,
  -0.0028391469,
  -0.0026884316,
  -0.019701406,
  0.008891215,
  0.019989394,
  0.023799125

In [167]:
len(embeddings)

201

In [170]:
embeddings[1]

[0.018671216,
 0.0002780376,
 0.0007606173,
 -0.039331224,
 0.010360835,
 -0.016888097,
 -0.014966768,
 -0.020261409,
 -0.00038796416,
 -0.009662948,
 -0.04148369,
 0.016161276,
 0.019301813,
 -0.037630267,
 0.032199856,
 0.007422137,
 0.020570891,
 0.018311802,
 -0.022159813,
 -0.016120855,
 -0.0043975124,
 0.081242755,
 -0.009628731,
 -0.013386655,
 -0.004424466,
 0.032191083,
 -0.0057068844,
 -0.004844922,
 -0.0036770627,
 0.0065424023,
 -0.021872219,
 0.002436384,
 0.008295372,
 0.020969559,
 -0.04489761,
 0.015141992,
 -0.008569029,
 0.0338257,
 0.03661317,
 0.006660987,
 -0.013596193,
 0.003833221,
 0.02057231,
 -0.004241438,
 -0.032663494,
 0.01621889,
 -0.01417731,
 0.003283915,
 0.0008784786,
 -0.0029777365,
 -0.0031290653,
 -0.0094849905,
 0.031658877,
 0.0038090306,
 -0.0037290119,
 -0.016848143,
 -0.009269105,
 -0.012604723,
 -0.016140463,
 -0.01667176,
 -0.015475159,
 0.011031658,
 0.020071404,
 0.015688457,
 -0.024330681,
 0.02666052,
 0.015359989,
 -0.0039209942,
 0.0104

In [171]:
len(embeddings[0])

4096

In [172]:
#STORING EMBEDDING BACK TO  DICTIONARY
for r, vec in zip(enriched_documents, embeddings):
    r["embedding"] = vec

print(f"embedded {len(enriched_documents)} chunks | vector dim = {len(enriched_documents[0]['embedding'])}")

embedded 201 chunks | vector dim = 4096


In [174]:
enriched_documents[2]

{'kind': 'prose',
 'chapter': 'Chapter 1: Tokenization',
 'section': '',
 'text': 'Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare words into reusable fragments.',
 'context': 'This chunk is from Chapter 1 (Tokenization) of "A Practitioner\'s Guide to Language Models," specifically discussing the evolution of tokenization methods from whitespace and character-based approaches to modern subword tokenization.',
 'enriched_text': 'This chunk is from Chapter 1 (Tokenization) of "A Practitioner\'s Guide to Language Models," specifically discussing the evolution of tokenization methods from whitespace and character-based approaches to modern subword t

## CREATING DATABASE

In [55]:
#STORING IN DATABASE EVERYTHING

import chromadb
import re


client = chromadb.PersistentClient(path="./chroma_db")
try: client.delete_collection("book")
except Exception: pass
collection = client.create_collection("book")

collection.add(
    ids=[f"chunk_{i}" for i in range(len(enriched_documents))],
    embeddings=[r["embedding"] for r in enriched_documents],
    documents=[r["enriched_text"] for r in enriched_documents],
    metadatas=[{                                     
        "original_chunk": r["text"],            
        "kind": r["kind"], "chapter": r["chapter"], "section": r["section"],
    } 
        for r in enriched_documents
    ]
)

print(f"stored {collection.count()} chunks in Chroma")

stored 201 chunks in Chroma


In [175]:
data = collection.get(
    include=["documents", "metadatas", "embeddings"]
)
print(data["embeddings"][1][:10])  # first 10 dimensions

[ 0.01867122  0.00027804  0.00076062 -0.03933122  0.01036084 -0.0168881
 -0.01496677 -0.02026141 -0.00038796 -0.00966295]


In [177]:
 data["documents"][1]

'This chunk is the introductory paragraph of Chapter 1: Tokenization in "A Practitioner\'s Guide to Language Models," establishing the fundamental role and downstream impact of converting text into tokens.\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.'

In [179]:
data["metadatas"][1]

{'kind': 'prose',
 'original_chunk': 'Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.',
 'chapter': 'Chapter 1: Tokenization',
 'section': ''}

In [232]:
# RETRIEVAL 
#query= "what are the main disadvantages of using character-level tokenization compared to subword tokenization?"
#query="Which vocabulary size is best suited for edge or mobile models, and what is its corresponding average tokens per word?"
#query = "What exact structural action is taken if an encoded token list exceeds the maximum allowable limit during safe tokenization?"
query = "What exact structural action is taken if an encoded token list exceeds the maximum allowable limit during safe tokenization? give me full code of it."

response = ollama.embed(model="qwen3-embedding:8b", input=query)
query_vector = response.embeddings[0]

# 2. Ask the database to find the 'k' closest matching chunks
search_results = collection.query(
    query_embeddings=[query_vector], 
    n_results=5
)


In [233]:
search_results

{'ids': [['chunk_22', 'chunk_21', 'chunk_20', 'chunk_16', 'chunk_19']],
 'embeddings': None,
 'documents': [['Chapter 1.3 Practical Notes: Final line of the `safe_tokenize` function used to bound input length for security and cost.\nreturn ids',
   'This code snippet is part of the `safe_tokenize` function in Chapter 1 (Tokenization), specifically within the "Practical Notes" section discussing security implications and the importance of bounding input length to control cost and latency.\nids = tokenizer.encode(text)\n    if len(ids) > max_tokens:\n        ids = ids[:max_tokens]            # truncate; or reject, per policy',
   'This code snippet is from Chapter 1 (Tokenization), specifically within the "Practical Notes" section, providing a function to implement hard length caps on tokenization to prevent excessive costs and latency in production.\ndef safe_tokenize(text, tokenizer, max_tokens=4096):\n    """Tokenize with a hard length cap to bound cost and latency."""',
   'Chapter 1

In [234]:
search_results["documents"]

[['Chapter 1.3 Practical Notes: Final line of the `safe_tokenize` function used to bound input length for security and cost.\nreturn ids',
  'This code snippet is part of the `safe_tokenize` function in Chapter 1 (Tokenization), specifically within the "Practical Notes" section discussing security implications and the importance of bounding input length to control cost and latency.\nids = tokenizer.encode(text)\n    if len(ids) > max_tokens:\n        ids = ids[:max_tokens]            # truncate; or reject, per policy',
  'This code snippet is from Chapter 1 (Tokenization), specifically within the "Practical Notes" section, providing a function to implement hard length caps on tokenization to prevent excessive costs and latency in production.\ndef safe_tokenize(text, tokenizer, max_tokens=4096):\n    """Tokenize with a hard length cap to bound cost and latency."""',
  'Chapter 1 (Tokenization), Section 1.2 (Special Tokens): Part of the `add_special` function used to append beginning-of-

In [235]:
len(search_results["documents"][0])

5

In [236]:
search_results["documents"][0][2]

'This code snippet is from Chapter 1 (Tokenization), specifically within the "Practical Notes" section, providing a function to implement hard length caps on tokenization to prevent excessive costs and latency in production.\ndef safe_tokenize(text, tokenizer, max_tokens=4096):\n    """Tokenize with a hard length cap to bound cost and latency."""'

## PASSING DOCUMENT TO LLM  FOR FINAL OUTPUT GENERATION.

In [237]:
# The documents field has the synthetic context line glued on ("This chunk introduces various sampling techniques..."). That was
# for retrieval. For generation you pull original_chunk from the metadata, or you'll be feeding model-written text into your
# answers.

In [238]:
def build_context(search_results):
    metadatas = search_results["metadatas"][0]
    tags = []
    for i, m in enumerate(metadatas, start=1):
        where = f"{m['chapter']} > {m['section']}".strip(" >")
        content = m["original_chunk"]
        tags.append(
            f'<document id="{i}">'
            f'\n ##source="{where}"'
            f'\n ###kind="{m["kind"]}">\n'
            f'{content}\n'
            f'</document id="{i}">'
        )
    return "\n\n".join(tags)

In [239]:
retrieved_content= build_context(search_results)
print(retrieved_content)

<document id="1">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
return ids
</document id="1">

<document id="2">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
ids = tokenizer.encode(text)
    if len(ids) > max_tokens:
        ids = ids[:max_tokens]            # truncate; or reject, per policy
</document id="2">

<document id="3">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
def safe_tokenize(text, tokenizer, max_tokens=4096):
    """Tokenize with a hard length cap to bound cost and latency."""
</document id="3">

<document id="4">
 ##source="Chapter 1: Tokenization > 1.2 Special Tokens"
 ###kind="code">
if eos: out = out + [SPECIAL_TOKENS["<eos>"]]
    return out
</document id="4">

<document id="5">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="prose">
Tokenization also has security implications. Adversarial inputs can exploit unusual token boundaries to bypass naive keyword f

In [240]:
retrieved_content

'<document id="1">\n ##source="Chapter 1: Tokenization > 1.3 Practical Notes"\n ###kind="code">\nreturn ids\n</document id="1">\n\n<document id="2">\n ##source="Chapter 1: Tokenization > 1.3 Practical Notes"\n ###kind="code">\nids = tokenizer.encode(text)\n    if len(ids) > max_tokens:\n        ids = ids[:max_tokens]            # truncate; or reject, per policy\n</document id="2">\n\n<document id="3">\n ##source="Chapter 1: Tokenization > 1.3 Practical Notes"\n ###kind="code">\ndef safe_tokenize(text, tokenizer, max_tokens=4096):\n    """Tokenize with a hard length cap to bound cost and latency."""\n</document id="3">\n\n<document id="4">\n ##source="Chapter 1: Tokenization > 1.2 Special Tokens"\n ###kind="code">\nif eos: out = out + [SPECIAL_TOKENS["<eos>"]]\n    return out\n</document id="4">\n\n<document id="5">\n ##source="Chapter 1: Tokenization > 1.3 Practical Notes"\n ###kind="prose">\nTokenization also has security implications. Adversarial inputs can exploit unusual token boun

## CALLING LLM FOR FINAL OUTPUT

In [241]:
final_doc = "<retrieved_documentation>\n\n" + retrieved_content + "\n</retrieved_documentation>"

In [242]:
print(final_doc)

<retrieved_documentation>

<document id="1">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
return ids
</document id="1">

<document id="2">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
ids = tokenizer.encode(text)
    if len(ids) > max_tokens:
        ids = ids[:max_tokens]            # truncate; or reject, per policy
</document id="2">

<document id="3">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
def safe_tokenize(text, tokenizer, max_tokens=4096):
    """Tokenize with a hard length cap to bound cost and latency."""
</document id="3">

<document id="4">
 ##source="Chapter 1: Tokenization > 1.2 Special Tokens"
 ###kind="code">
if eos: out = out + [SPECIAL_TOKENS["<eos>"]]
    return out
</document id="4">

<document id="5">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="prose">
Tokenization also has security implications. Adversarial inputs can exploit unusual token boundarie

In [243]:
#FINAL PROMPT 

Final_prompt= "\n\n".join([final_doc, f"<user_question>{query}</user_question>"])

In [244]:
print(Final_prompt)

<retrieved_documentation>

<document id="1">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
return ids
</document id="1">

<document id="2">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
ids = tokenizer.encode(text)
    if len(ids) > max_tokens:
        ids = ids[:max_tokens]            # truncate; or reject, per policy
</document id="2">

<document id="3">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="code">
def safe_tokenize(text, tokenizer, max_tokens=4096):
    """Tokenize with a hard length cap to bound cost and latency."""
</document id="3">

<document id="4">
 ##source="Chapter 1: Tokenization > 1.2 Special Tokens"
 ###kind="code">
if eos: out = out + [SPECIAL_TOKENS["<eos>"]]
    return out
</document id="4">

<document id="5">
 ##source="Chapter 1: Tokenization > 1.3 Practical Notes"
 ###kind="prose">
Tokenization also has security implications. Adversarial inputs can exploit unusual token boundarie

In [245]:
#system instruction
system_instructions= """
<system_role>
You are a factual, highly accurate Retrieval-Augmented Generation (RAG) assistant. 
Your primary function is to answer user queries by synthesizing information exclusively from the retrieved passages provided in the context. 
</system_role>

<input_format>
You will receive inputs containing two distinct parts, separated by XML tags:
1. <retrieved_documentation>: This section contains the external corpus passages, chunks, and associated metadata (e.g., source document, section, position) retrieved to help answer the user's query.
2. <user_question>: This section contains the specific query or prompt you need to address.
</input_format>

<core_instructions>
1. Grounding: You must base your answers strictly on the external corpus passages provided within the <retrieved_documentation> tags. Do not rely on your frozen internal training data or outside knowledge.
2. Context Dependence: Because the quality of retrieval sets a ceiling on the quality of the answer, you must only answer if the necessary information is explicitly present in the provided documentation. 
3. Hallucination Prevention: If the right chunk is never retrieved, no amount of prompt engineering will recover the answer. Therefore, if the <retrieved_documentation> does not contain the answer, explicitly state: "I cannot answer this question based on the provided context." Do not guess or infer outside the text.
4. Citations: You must cite your sources in your answer.such as chapter name , section name etc. found within the <retrieved_documentation> tags to accurately reference exactly where you found the information. 
5. Formatting: Keep structured elements like code blocks, lists, and tables intact to preserve meaning. Use markdown formatting to ensure your response is clear and readable.
</core_instructions>

<response_guidelines>
- Read the <user_question> carefully.
- Scan the <retrieved_documentation> for relevant facts.
- Formulate a direct, concise, and accurate response.
- Append citations directly after the sentence or claim they support.
</response_guidelines>"""

In [246]:
response = ollama.generate(
    model='gemma4:31b-cloud',
    prompt=Final_prompt,
    system=system_instructions)

In [247]:
print(response.response)

If an encoded token list exceeds the maximum allowable limit during safe tokenization, the list is truncated to the maximum number of tokens (or rejected, depending on the policy) [Chapter 1: Tokenization > 1.3 Practical Notes].

The full code for this process is as follows:

```python
def safe_tokenize(text, tokenizer, max_tokens=4096):
    """Tokenize with a hard length cap to bound cost and latency."""
    ids = tokenizer.encode(text)
    if len(ids) > max_tokens:
        ids = ids[:max_tokens]            # truncate; or reject, per policy
    return ids
```
[Chapter 1: Tokenization > 1.3 Practical Notes]


In [63]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(allow_reset=True)
)

client.reset()
print("Database has been completely wiped clean.")

ValueError: An instance of Chroma already exists for ./chroma_db with different settings